# 🧬 GA Feature Selection — LOSO CV
### 3 Dataset × 3 Classifier (SVM · RF · NB)

**Struktur eksperimen:**
- Dataset   : MFCC | LPCC | COMBINED
- Classifier: LinearSVC | RandomForest | GaussianNB
- CV Method : Leave-One-Subject-Out (LOSO / cross-subject)
- Total     : **9 eksperimen** (3 dataset × 3 classifier)
- Parameter GA fixed:
  - Population size: **50**
  - Generation: **50**
  - Crossover rate: **0.8**
  - Mutation rate: **0.05**
  - Alpha feature penalty: **0.01**

Notebook ini dibuat dari konsep GA lama + struktur eksperimen PSO 3 model/3 dataset.


## ✅ Auto Resume / Auto Skip

Notebook ini punya dua mekanisme lanjut jalan:

1. **Checkpoint per eksperimen** (`.pkl`)  
   Jika notebook berhenti di tengah generasi, eksperimen bisa resume dari generasi terakhir yang tersimpan.

2. **Live results CSV** (`ga_results_live.csv`)  
   Jika eksperimen seperti `MFCC_SVM` sudah selesai dengan parameter GA yang sama, eksperimen akan otomatis di-skip.


## 📁 Cell 1 — Load Data


In [ ]:
import pandas as pd
import os

# Auto-detect path: lokal (Windows) atau Codespaces/Linux
_BASE = 'D:/LISA/.retest_new' if os.path.exists('D:/LISA/.retest_new/MFCC_new.csv') else os.getcwd()
MFCC_PATH = os.path.join(_BASE, 'MFCC_new.csv')
LPCC_PATH = os.path.join(_BASE, 'LPCC_new.csv')

mfcc_df = pd.read_csv(MFCC_PATH)
lpcc_df = pd.read_csv(LPCC_PATH)

print('MFCC shape :', mfcc_df.shape)
print('LPCC shape :', lpcc_df.shape)

assert 'filename' in mfcc_df.columns, "Kolom 'filename' tidak ditemukan di MFCC!"
assert 'filename' in lpcc_df.columns, "Kolom 'filename' tidak ditemukan di LPCC!"
assert 'label' in mfcc_df.columns, "Kolom 'label' tidak ditemukan di MFCC!"
assert 'label' in lpcc_df.columns, "Kolom 'label' tidak ditemukan di LPCC!"
assert 'datatype' in mfcc_df.columns, "Kolom 'datatype' tidak ditemukan di MFCC!"
assert 'datatype' in lpcc_df.columns, "Kolom 'datatype' tidak ditemukan di LPCC!"

print('\nContoh filename MFCC (5 pertama):')
print(mfcc_df['filename'].head().tolist())
print('\n✅ Cell 1 OK — Data loaded')


## ⚙️ Cell 2 — Import & Konfigurasi Global


In [ ]:
import numpy as np
import pandas as pd
import os
import time
import gc
import pickle
from datetime import datetime
from collections import OrderedDict
from threading import Lock
import pytz

from sklearn.metrics import f1_score
from sklearn.base import clone
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from joblib import Parallel, delayed

try:
    from threadpoolctl import threadpool_limits
except Exception:
    threadpool_limits = None

# Seed untuk reproduksi hasil
RANDOM_STATE = 42
STATE_VERSION = 11  # dinaikkan: format cache key (packbits) + signature fitness berubah
np.random.seed(RANDOM_STATE)

WIB = pytz.timezone('Asia/Jakarta')

# Parallel worker
CPU_COUNT = os.cpu_count() or 1
GA_N_JOBS = max(1, CPU_COUNT - 1)

# Folder output
BASE_DIR = 'D:/LISA/.retest_new/Data_Output_GA_3Model_LOSO_Scaled'
os.makedirs(BASE_DIR, exist_ok=True)

# PARAMETER GA — fixed
POP_SIZE = 50
N_GEN    = 50
CR       = 0.8
MR       = 0.05
ALPHA    = 0.01
N_ELITE  = 2
SAVE_EVERY = 25

# CLASSIFIERS
#   SVM : LinearSVC murni (tanpa kalibrasi). Hanya .predict() yang dipakai untuk F1.
#         tol=1e-2 agar konvergen cepat pada data ter-scale.
#   RF  : Random Forest, n_jobs=1 karena paralel utama ada di GA.
#   NB  : GaussianNB.
CLASSIFIERS = {
    'SVM': LinearSVC(random_state=RANDOM_STATE, max_iter=5000, tol=1e-2),
    'RF' : RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    'NB' : GaussianNB(),
}

def format_time(sec):
    m = int(sec // 60)
    s = int(sec % 60)
    return f'{m}m {s}s ({sec:.1f}s)'

def get_timestamp_wib():
    return datetime.now(WIB).strftime('%H:%M:%S')

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

print(f'CPU count  : {CPU_COUNT}')
print(f'GA_N_JOBS  : {GA_N_JOBS}')
print(f'Output dir : {BASE_DIR}')
print('Cell 2 OK — Config & Classifiers siap')
print(f'   Classifier: {list(CLASSIFIERS.keys())}')
print(f'   POP_SIZE={POP_SIZE} | N_GEN={N_GEN} | CR={CR} | MR={MR} | ALPHA={ALPHA}')


## 📦 Cell 3 — Persiapan Data


In [ ]:
def extract_subject_from_filename(filename):
    """
    Ekstrak label subjek dari nama file.
    Format file: SPEAKER_BATCH_KONDISI_MIC.wav
    Contoh     : CF02_B1_D0_M2.wav -> 'CF02'

    Catatan:
    - Dibuat sama seperti notebook PSO: token pertama sebelum underscore.
    - Dengan LOSO, semua sampel dari subjek yang sama full masuk train atau validasi.
    """
    basename = os.path.basename(str(filename)).replace('.wav', '')
    parts = basename.split('_')
    if len(parts) < 1 or parts[0] == '':
        raise ValueError(f"Format filename tidak dikenali: '{filename}'")
    return parts[0]


def clean_dataframe(df):
    """Bersihkan dataset: drop NaN, reset index, pastikan label integer."""
    df = df.dropna().reset_index(drop=True).copy()
    df['label'] = df['label'].astype(int)
    return df


def split_train_test(df):
    """Split berdasarkan kolom datatype: train dan test."""
    train_df = df[df['datatype'].astype(str).str.lower() == 'train'].reset_index(drop=True)
    test_df  = df[df['datatype'].astype(str).str.lower() == 'test'].reset_index(drop=True)
    if train_df.empty:
        raise ValueError("Data train kosong. Cek isi kolom datatype == 'train'.")
    if test_df.empty:
        raise ValueError("Data test kosong. Cek isi kolom datatype == 'test'.")
    return train_df, test_df


def prepare_features(df):
    """
    Pisahkan DataFrame menjadi:
      X        : array fitur float32
      y        : array label integer
      subjects : array label subjek per baris

    Kolom non-fitur yang dibuang: Unnamed: 0, filename, subject, datatype, label
    """
    subjects = df['filename'].apply(extract_subject_from_filename).values
    drop_cols = [c for c in ['Unnamed: 0', 'filename', 'subject', 'datatype', 'label'] if c in df.columns]
    X = df.drop(columns=drop_cols).values.astype(np.float32)
    y = df['label'].values.astype(int)
    return X, y, subjects


def prepare_all_datasets(mfcc_df, lpcc_df):
    """
    Siapkan semua dataset:
      MFCC, LPCC, COMBINED
    Return dict:
      {name: (X_tr, X_te, y_tr, y_te, subjects_tr)}
    """
    mfcc_clean = clean_dataframe(mfcc_df)
    lpcc_clean = clean_dataframe(lpcc_df)

    mfcc_tr, mfcc_te = split_train_test(mfcc_clean)
    lpcc_tr, lpcc_te = split_train_test(lpcc_clean)

    if len(mfcc_tr) != len(lpcc_tr) or len(mfcc_te) != len(lpcc_te):
        raise ValueError(
            'Jumlah baris train/test MFCC dan LPCC tidak sama. '
            'COMBINED butuh urutan data yang sejajar.'
        )

    X_mfcc_tr, y_tr, subjects_mfcc = prepare_features(mfcc_tr)
    X_mfcc_te, y_te, _             = prepare_features(mfcc_te)
    X_lpcc_tr, y_lpcc_tr, subjects_lpcc = prepare_features(lpcc_tr)
    X_lpcc_te, y_lpcc_te, _             = prepare_features(lpcc_te)

    if not np.array_equal(y_tr, y_lpcc_tr):
        raise ValueError('Label train MFCC dan LPCC tidak sejajar. Cek urutan data.')
    if not np.array_equal(y_te, y_lpcc_te):
        raise ValueError('Label test MFCC dan LPCC tidak sejajar. Cek urutan data.')

    # ── Opsi A: StandardScaler di-fit SEKALI di train, lalu transform train+test ──
    # Dilakukan di luar loop CV (sekali saja) → biaya hampir nol, tapi LinearSVC/NB
    # jadi ter-scale sehingga konvergen normal (hilang ConvergenceWarning).
    # RF tidak terpengaruh scaling, jadi aman dipakai untuk semua classifier.
    from sklearn.preprocessing import StandardScaler
    _sc_mfcc = StandardScaler().fit(X_mfcc_tr)
    X_mfcc_tr = _sc_mfcc.transform(X_mfcc_tr).astype(np.float32)
    X_mfcc_te = _sc_mfcc.transform(X_mfcc_te).astype(np.float32)
    _sc_lpcc = StandardScaler().fit(X_lpcc_tr)
    X_lpcc_tr = _sc_lpcc.transform(X_lpcc_tr).astype(np.float32)
    X_lpcc_te = _sc_lpcc.transform(X_lpcc_te).astype(np.float32)

    X_comb_tr = np.concatenate([X_mfcc_tr, X_lpcc_tr], axis=1)
    X_comb_te = np.concatenate([X_mfcc_te, X_lpcc_te], axis=1)

    datasets = {
        'MFCC'    : (X_mfcc_tr, X_mfcc_te, y_tr, y_te, subjects_mfcc),
        'LPCC'    : (X_lpcc_tr, X_lpcc_te, y_tr, y_te, subjects_lpcc),
        'COMBINED': (X_comb_tr, X_comb_te, y_tr, y_te, subjects_mfcc),
    }

    print('\n📊 Distribusi subjek di training set (MFCC):')
    unique_s, counts_s = np.unique(subjects_mfcc, return_counts=True)
    for s, c in zip(unique_s, counts_s):
        print(f'   {s}: {c} sampel')
    print(f'   -> Leave-One-Subject-Out membuat {len(unique_s)} fold per eksperimen')

    print('\n📐 Ukuran fitur per dataset:')
    for name, (Xtr, Xte, ytr, yte, subj) in datasets.items():
        print(f'   {name:<10}: train={Xtr.shape}, test={Xte.shape}, subjects={len(np.unique(subj))}')

    return datasets


datasets = prepare_all_datasets(mfcc_df, lpcc_df)
print('\n✅ Cell 3 OK — Dataset siap')


## 💾 Cell 4 — Fitness Cache & Checkpoint


In [ ]:
# FITNESS CACHE + CHECKPOINT
# Format cache: { individu_biner_packed_bytes: (fitness, f1_mean, f1_std) }
MAX_CACHE_SIZE = 5000
fitness_cache = OrderedDict()
_cache_lock = Lock()

def make_cache_key(ind):
    # Pack biner ke byte (8 bit per byte) — ~8x lebih kecil dari .tobytes() langsung
    arr = np.packbits(np.asarray(ind, dtype=np.uint8))
    return arr.tobytes()

def reset_cache():
    global fitness_cache
    with _cache_lock:
        fitness_cache = OrderedDict()
    gc.collect()
    print('Cache direset')

def get_cache(key):
    with _cache_lock:
        value = fitness_cache.get(key)
        if value is not None:
            fitness_cache.move_to_end(key)
        return value

def set_cache(key, value):
    with _cache_lock:
        if key in fitness_cache:
            fitness_cache.move_to_end(key)
            fitness_cache[key] = value
            return
        if len(fitness_cache) >= MAX_CACHE_SIZE:
            fitness_cache.popitem(last=False)
        fitness_cache[key] = value

def get_cached_stats(ind):
    cached = get_cache(make_cache_key(ind))
    return cached if cached is not None else (0.0, 0.0, 0.0)


def save_state(filename, state):
    try:
        with open(filename, 'wb') as f:
            pickle.dump(state, f)
    except Exception as e:
        print(f'Gagal simpan checkpoint: {e}')


def load_state(filename):
    try:
        with open(filename, 'rb') as f:
            state = pickle.load(f)
    except Exception as e:
        raise RuntimeError(f"Gagal load checkpoint '{filename}': {e}")

    if state.get('version') != STATE_VERSION:
        raise ValueError(
            f"State version mismatch: file={state.get('version')}, code={STATE_VERSION}. "
            f"Hapus checkpoint lama jika ingin mulai ulang."
        )

    if 'np_random_state' in state:
        np.random.set_state(state['np_random_state'])
    return state

print('Cell 4 OK — Cache (packed key) & Checkpoint siap')


## 🧬 Cell 5 — Evaluasi LOSO, Fitness, Operator GA


In [ ]:
def make_loso_folds(subjects):
    subjects = np.asarray(subjects)
    folds = []
    for s in np.unique(subjects):
        val_idx = np.flatnonzero(subjects == s)
        tr_idx  = np.flatnonzero(subjects != s)
        if len(tr_idx) > 0 and len(val_idx) > 0:
            folds.append((tr_idx, val_idx))
    return folds


def make_loso_fold_arrays(X, y, folds):
    \"\"\"Pre-slice X dan y per fold SEKALI di awal. Return list (X_tr, X_val, y_tr, y_val).\"\"\"
    result = []
    for tr_idx, val_idx in folds:
        result.append((
            np.ascontiguousarray(X[tr_idx]),
            np.ascontiguousarray(X[val_idx]),
            y[tr_idx],
            y[val_idx],
        ))
    return result


def evaluate_model_presliced(fold_arrays, model, feature_idx):
    \"\"\"Evaluasi LOSO dengan fold pre-sliced. Indexing kolom saja (murah).\"\"\"
    scores = []
    for X_tr, X_val, y_tr, y_val in fold_arrays:
        m = clone(model)
        if feature_idx is None:
            m.fit(X_tr, y_tr)
            pred = m.predict(X_val)
        else:
            m.fit(X_tr[:, feature_idx], y_tr)
            pred = m.predict(X_val[:, feature_idx])
        scores.append(f1_score(y_val, pred, average='weighted'))
    if not scores:
        return 0.0, 0.0
    return float(np.mean(scores)), float(np.std(scores))


def fitness_function(ind, fold_arrays, model, alpha, n_feat_total):
    \"\"\"Fitness GA: F1_mean_LOSO - alpha * (n_fitur / n_fitur_total).\"\"\"
    key = make_cache_key(ind)
    cached = get_cache(key)
    if cached is not None:
        return cached[0]

    idx = np.flatnonzero(ind)
    if len(idx) == 0:
        set_cache(key, (0.0, 0.0, 0.0))
        return 0.0

    mean_f1, std_f1 = evaluate_model_presliced(fold_arrays, model, idx)
    fit = mean_f1 - alpha * (len(idx) / n_feat_total)
    set_cache(key, (fit, mean_f1, std_f1))
    return fit


def create_population(size, n_feat):
    pop = np.random.randint(0, 2, (size, n_feat), dtype=np.uint8)
    empty = np.flatnonzero(pop.sum(axis=1) == 0)
    for i in empty:
        pop[i, np.random.randint(n_feat)] = 1
    return pop


def selection_with_elitism(pop, fitness, n_elite=2):
    sorted_idx = np.argsort(fitness)
    elite = pop[sorted_idx[-n_elite:]].copy()
    selected = pop[sorted_idx[-len(pop)//2:]].copy()
    return selected, elite


def crossover_2point(p1, p2):
    n = len(p1)
    if n < 4:
        child = p1.copy()
    else:
        pt1 = np.random.randint(1, n - 2)
        pt2 = np.random.randint(pt1 + 1, n - 1)
        child = np.concatenate([p1[:pt1], p2[pt1:pt2], p1[pt2:]]).astype(np.uint8)
    if child.sum() == 0:
        child[np.random.randint(n)] = 1
    return child


def mutate(ind, rate):
    ind = ind.copy()
    mask = np.random.rand(len(ind)) < rate
    ind[mask] = 1 - ind[mask]
    if ind.sum() == 0:
        ind[np.random.randint(len(ind))] = 1
    return ind.astype(np.uint8)

print('Cell 5 OK — Evaluasi LOSO pre-sliced, Fitness, Operator GA siap')


## 🔁 Cell 6 — Fungsi Utama GA


In [ ]:
def genetic_algorithm_feature_selection(
    X, y, subjects, model,
    pop_size=POP_SIZE, n_gen=N_GEN, cr=CR, mr=MR, alpha=ALPHA,
    ckpt_file=None, gen_log_file=None,
    n_jobs=None, n_elite=N_ELITE
):
    \"\"\"GA feature selection dengan LOSO CV. Return (best_ind, best_fitness, f1_mean, f1_std).\"\"\"
    global fitness_cache

    X = np.ascontiguousarray(X)
    y = np.asarray(y)
    subjects = np.asarray(subjects)
    n_feat = X.shape[1]

    # n_jobs per-classifier: SVM lebih efisien di worker lebih sedikit
    if n_jobs is None:
        if type(model).__name__ == 'LinearSVC':
            n_jobs_eval = min(6, max(1, os.cpu_count() - 1))
        else:
            n_jobs_eval = max(1, os.cpu_count() - 1)
    else:
        n_jobs_eval = n_jobs

    loso_folds = make_loso_folds(subjects)
    if len(loso_folds) < 2:
        raise ValueError('LOSO butuh minimal 2 subjek pada data training.')

    # Pre-slice fold sekali — penghematan terbesar
    fold_arrays = make_loso_fold_arrays(X, y, loso_folds)

    # Resume / Init
    if ckpt_file and os.path.exists(ckpt_file):
        state = load_state(ckpt_file)
        pop = state['pop']
        start_gen = int(state['gen']) + 1
        fitness_cache = state.get('fitness_cache', OrderedDict())
        gen_history = state.get('gen_history', [])
        best_global = state.get('best_global', -np.inf)
        best_global_ind = state.get('best_global_ind', None)
        elapsed_before = float(state.get('elapsed_before', 0.0))
        print(f'Resume dari generasi {start_gen + 1}/{n_gen} | waktu sebelumnya {format_time(elapsed_before)}')
    else:
        pop = create_population(pop_size, n_feat)
        start_gen = 0
        gen_history = []
        best_global = -np.inf
        best_global_ind = None
        elapsed_before = 0.0

    session_start = time.time()

    for gen in range(start_gen, n_gen):
        gen_start = time.time()
        ts = get_timestamp_wib()

        try:
            if threadpool_limits is not None:
                with threadpool_limits(limits=1):
                    fitness = np.array(
                        Parallel(n_jobs=n_jobs_eval, prefer='threads', batch_size='auto')(
                            delayed(fitness_function)(ind, fold_arrays, model, alpha, n_feat)
                            for ind in pop
                        ), dtype=np.float64
                    )
            else:
                fitness = np.array(
                    Parallel(n_jobs=n_jobs_eval, prefer='threads', batch_size='auto')(
                        delayed(fitness_function)(ind, fold_arrays, model, alpha, n_feat)
                        for ind in pop
                    ), dtype=np.float64
                )
        except Exception as e:
            print(f'Parallel gagal gen {gen+1}, fallback sequential: {e}')
            fitness = np.array([
                fitness_function(ind, fold_arrays, model, alpha, n_feat)
                for ind in pop
            ], dtype=np.float64)

        best_idx = int(np.argmax(fitness))
        best_fit_val = float(fitness[best_idx])
        best_ind_gen = pop[best_idx].copy()
        _, best_f1mean, best_f1std = get_cached_stats(best_ind_gen)

        if best_fit_val > best_global:
            best_global = best_fit_val
            best_global_ind = best_ind_gen.copy()

        n_feat_sel = int(best_ind_gen.sum())
        gen_time = time.time() - gen_start
        session_elapsed = time.time() - session_start
        total_elapsed = elapsed_before + session_elapsed
        gens_done = gen + 1
        gens_remaining = n_gen - gens_done
        avg_per_gen = total_elapsed / gens_done if gens_done > 0 else 0
        eta = avg_per_gen * gens_remaining

        print(
            f'[{ts} WIB] Gen {gen+1:>3}/{n_gen} | '
            f'Best={best_fit_val:.4f} | Mean={np.mean(fitness):.4f} | Std={np.std(fitness):.4f} | '
            f'F1cv={best_f1mean:.4f}+/-{best_f1std:.4f} | '
            f'Feat={n_feat_sel}/{n_feat} | Cache={len(fitness_cache)} | '
            f'Time={format_time(gen_time)} | Elapsed={format_time(total_elapsed)} | ETA={format_time(eta)}'
        )

        gen_history.append({
            'gen'           : gen + 1,
            'timestamp_wib' : ts,
            'best_fitness'  : best_fit_val,
            'mean_fitness'  : float(np.mean(fitness)),
            'std_fitness'   : float(np.std(fitness)),
            'best_f1_mean'  : best_f1mean,
            'best_f1_std'   : best_f1std,
            'n_features'    : n_feat_sel,
            'cache_size'    : len(fitness_cache),
            'gen_time_s'    : round(gen_time, 2),
            'total_time_s'  : round(total_elapsed, 2),
        })

        selected, elite = selection_with_elitism(pop, fitness, n_elite=n_elite)
        next_pop = []
        while len(next_pop) < pop_size - len(elite):
            p1, p2 = selected[np.random.randint(len(selected), size=2)]
            child = crossover_2point(p1, p2) if np.random.rand() < cr else p1.copy()
            child = mutate(child, mr)
            next_pop.append(child)

        for e in elite:
            next_pop.append(e)
        pop = np.array(next_pop, dtype=np.uint8)

        if (gen + 1) % SAVE_EVERY == 0:
            if gen_log_file:
                try:
                    pd.DataFrame(gen_history).to_csv(gen_log_file, index=False)
                except Exception as e:
                    print(f'Gagal tulis gen log: {e}')

            if ckpt_file:
                state = {
                    'version'         : STATE_VERSION,
                    'gen'             : gen,
                    'pop'             : pop,
                    'fitness_cache'   : fitness_cache,
                    'gen_history'     : gen_history,
                    'best_global'     : best_global,
                    'best_global_ind' : best_global_ind,
                    'elapsed_before'  : total_elapsed,
                    'np_random_state' : np.random.get_state(),
                    'done'            : False,
                }
                save_state(ckpt_file, state)

    if best_global_ind is None:
        final_fitness = np.array([
            fitness_function(ind, fold_arrays, model, alpha, n_feat)
            for ind in pop
        ], dtype=np.float64)
        best_idx = int(np.argmax(final_fitness))
        best_global_ind = pop[best_idx].copy()
        best_global = float(final_fitness[best_idx])

    _, best_f1mean, best_f1std = get_cached_stats(best_global_ind)
    total_elapsed = elapsed_before + (time.time() - session_start)

    if ckpt_file:
        final_state = {
            'version'         : STATE_VERSION,
            'gen'             : n_gen - 1,
            'pop'             : pop,
            'fitness_cache'   : fitness_cache,
            'gen_history'     : gen_history,
            'best_global'     : best_global,
            'best_global_ind' : best_global_ind,
            'elapsed_before'  : total_elapsed,
            'np_random_state' : np.random.get_state(),
            'done'            : True,
        }
        save_state(ckpt_file, final_state)

    if gen_log_file:
        try:
            pd.DataFrame(gen_history).to_csv(gen_log_file, index=False)
        except Exception as e:
            print(f'Gagal tulis final gen log: {e}')

    return best_global_ind, float(best_global), best_f1mean, best_f1std

print('Cell 6 OK — Fungsi utama GA siap (pre-sliced)')


## 🏃 Cell 7 — Runner Eksperimen (3 Dataset × 3 Classifier)


In [ ]:
# ──────────────────────────────────────────────────────
# RUNNER UTAMA + AUTO RESUME / AUTO SKIP
# ──────────────────────────────────────────────────────

datasets_to_run = {k: datasets[k] for k in ['MFCC', 'LPCC', 'COMBINED']}
clf_to_run = {k: CLASSIFIERS[k] for k in ['SVM', 'RF', 'NB']}

LIVE_RESULTS_PATH  = os.path.join(BASE_DIR, 'ga_results_live.csv')
FINAL_RESULTS_PATH = os.path.join(BASE_DIR, 'ga_results_final.csv')

# ── Load hasil live lama jika ada ──
if os.path.exists(LIVE_RESULTS_PATH):
    df_live = pd.read_csv(LIVE_RESULTS_PATH)
    if 'exp_id' in df_live.columns:
        df_live = df_live.drop_duplicates(subset=['exp_id'], keep='last')
    results = df_live.to_dict('records')
    print(f'🔁 Resume aktif: ditemukan {len(results)} hasil lama dari ga_results_live.csv')
else:
    df_live = pd.DataFrame()
    results = []
    print('🆕 Tidak ada ga_results_live.csv lama — mulai eksperimen dari awal')


def is_experiment_completed(df_live, exp_id):
    """Cek apakah exp_id sudah selesai dengan parameter GA yang sama."""
    if df_live.empty or 'exp_id' not in df_live.columns:
        return False

    row = df_live[df_live['exp_id'].astype(str) == str(exp_id)]
    if row.empty:
        return False

    row = row.iloc[-1]
    checks = {
        'pop_size': POP_SIZE,
        'n_gen'   : N_GEN,
        'cr'      : CR,
        'mr'      : MR,
        'alpha'   : ALPHA,
    }

    for col, current_value in checks.items():
        if col not in row.index:
            return False
        try:
            if float(row[col]) != float(current_value):
                return False
        except Exception:
            if row[col] != current_value:
                return False
    return True


def save_live_results(results):
    """Simpan live CSV dengan aman dan drop duplikat exp_id."""
    df_tmp = pd.DataFrame(results)
    if not df_tmp.empty and 'exp_id' in df_tmp.columns:
        df_tmp = df_tmp.drop_duplicates(subset=['exp_id'], keep='last')
    df_tmp.to_csv(LIVE_RESULTS_PATH, index=False)
    return df_tmp


runner_start = time.time()
total = len(datasets_to_run) * len(clf_to_run)
counter = 0
skipped = 0
executed = 0

print('=' * 80)
print('🚀 GA FEATURE SELECTION — MULAI')
print(f'   Dataset    : {list(datasets_to_run.keys())}')
print(f'   Classifier : {list(clf_to_run.keys())}')
print(f'   Total      : {total} eksperimen')
print(f'   POP_SIZE={POP_SIZE} | N_GEN={N_GEN} | CR={CR} | MR={MR} | ALPHA={ALPHA}')
print('=' * 80)

for ds_name, (X_tr, X_te, y_tr, y_te, subjects_tr) in datasets_to_run.items():

    for clf_name, clf in clf_to_run.items():
        reset_cache()
        counter += 1
        exp_id = f'{ds_name}_{clf_name}'

        dataset_dir = os.path.join(BASE_DIR, ds_name)
        ensure_dir(dataset_dir)
        ckpt_file = os.path.join(dataset_dir, f'{exp_id}.pkl')
        gen_log_file = os.path.join(dataset_dir, f'{exp_id}_genlog.csv')

        # ── AUTO SKIP jika live result sudah ada dengan parameter sama ──
        if is_experiment_completed(df_live, exp_id):
            skipped += 1
            print(f'\n⏭️  SKIP [{counter}/{total}] {exp_id}')
            print('   Alasan: hasil sudah ada di ga_results_live.csv dan parameter GA sama')
            continue

        # ── Jika checkpoint sudah done tapi live CSV belum ada, pakai checkpoint untuk evaluasi ulang test ──
        resume_from_done_ckpt = False
        if os.path.exists(ckpt_file):
            try:
                state = load_state(ckpt_file)
                if state.get('done', False) and state.get('best_global_ind') is not None:
                    print(f'\n📦 Checkpoint done ditemukan untuk {exp_id}; evaluasi ulang test dan simpan ke live CSV')
                    best_ind = state['best_global_ind']
                    best_fit = float(state.get('best_global', 0.0))
                    reset_cache()
                    loso_folds = make_loso_folds(subjects_tr)
                    fold_arrays = make_loso_fold_arrays(X_tr, y_tr, loso_folds)
                    # isi ulang cache untuk statistik CV dari best_ind
                    _ = fitness_function(best_ind, fold_arrays, clf, ALPHA, X_tr.shape[1])
                    _, f1_cv_mean, f1_cv_std = get_cached_stats(best_ind)
                    resume_from_done_ckpt = True
            except Exception as e:
                print(f'⚠️  Checkpoint tidak kompatibel/rusak untuk {exp_id}, mulai ulang: {e}')
                try:
                    os.remove(ckpt_file)
                except Exception as remove_err:
                    print(f'⚠️  Gagal hapus checkpoint lama: {remove_err}')

        executed += 1
        exp_start = time.time()
        ts_start = get_timestamp_wib()

        print(f'\n{"=" * 80}')
        print(f'🟢 [{counter}/{total}] {exp_id}')
        print(f'   Dataset   : {ds_name} ({X_tr.shape[1]} fitur, {X_tr.shape[0]} sampel)')
        print(f'   Classifier: {clf_name}')
        print(f'   GA        : POP={POP_SIZE} | GEN={N_GEN} | CR={CR} | MR={MR} | ALPHA={ALPHA}')
        print(f'   CV        : Leave-One-Subject-Out ({len(np.unique(subjects_tr))} fold)')
        print(f'   Mulai     : {ts_start} WIB')
        print(f'{"=" * 80}')

        # ── Jalankan GA jika belum ada checkpoint done ──
        if not resume_from_done_ckpt:
            best_ind, best_fit, f1_cv_mean, f1_cv_std = genetic_algorithm_feature_selection(
                X_tr, y_tr, subjects_tr, clf,
                pop_size=POP_SIZE,
                n_gen=N_GEN,
                cr=CR,
                mr=MR,
                alpha=ALPHA,
                ckpt_file=ckpt_file,
                gen_log_file=gen_log_file,
                n_jobs=GA_N_JOBS,
                n_elite=N_ELITE,
            )

        # ── Evaluasi test set ──
        selected_idx = np.flatnonzero(best_ind)
        n_selected = len(selected_idx)

        if n_selected == 0:
            raise RuntimeError(f'{exp_id}: tidak ada fitur terpilih. Ini seharusnya tidak terjadi karena ada guard.')

        model_test = clone(clf)
        model_test.fit(X_tr[:, selected_idx], y_tr)
        pred_te = model_test.predict(X_te[:, selected_idx])
        f1_test = float(f1_score(y_te, pred_te, average='weighted'))

        exp_time = time.time() - exp_start
        ts_end = get_timestamp_wib()

        print(f'\n✅ SELESAI [{counter}/{total}] {exp_id}')
        print(f'   Selesai       : {ts_end} WIB')
        print(f'   Durasi        : {format_time(exp_time)}')
        print(f'   F1 Test       : {f1_test:.4f}')
        print(f'   F1 CV (LOSO)  : {f1_cv_mean:.4f} ± {f1_cv_std:.4f}')
        print(f'   Gap CV<->Test : {abs(f1_cv_mean - f1_test):.4f}')
        print(f'   Fitur terpilih: {n_selected}/{X_tr.shape[1]} ({100*n_selected/X_tr.shape[1]:.1f}%)')

        results.append({
            'exp_id'          : exp_id,
            'dataset'         : ds_name,
            'classifier'      : clf_name,
            'cv_method'       : 'LeaveOneSubjectOut',
            'f1_test'         : round(f1_test, 4),
            'f1_cv_mean'      : round(f1_cv_mean, 4),
            'f1_cv_std'       : round(f1_cv_std, 4),
            'f1_gap'          : round(abs(f1_cv_mean - f1_test), 4),
            'best_fitness'    : round(best_fit, 4),
            'n_feat_selected' : n_selected,
            'n_feat_total'    : X_tr.shape[1],
            'feat_ratio_pct'  : round(100 * n_selected / X_tr.shape[1], 2),
            'duration_s'      : round(exp_time, 2),
            'pop_size'        : POP_SIZE,
            'n_gen'           : N_GEN,
            'cr'              : CR,
            'mr'              : MR,
            'alpha'           : ALPHA,
            'time_start_wib'  : ts_start,
            'time_end_wib'    : ts_end,
        })

        df_live = save_live_results(results)

# ──────────────────────────────────────────────────────
# HASIL AKHIR
# ──────────────────────────────────────────────────────
df_results = pd.DataFrame(results)

if df_results.empty:
    print('\n⚠️ Tidak ada hasil eksperimen untuk ditampilkan.')
else:
    if 'exp_id' in df_results.columns:
        df_results = df_results.drop_duplicates(subset=['exp_id'], keep='last')

    df_results = (
        df_results
        .sort_values('f1_test', ascending=False)
        .reset_index(drop=True)
    )
    df_results.to_csv(FINAL_RESULTS_PATH, index=False)

    total_time = time.time() - runner_start
    print(f'\n{"=" * 80}')
    print(f'🏁 SEMUA EKSPERIMEN SELESAI — {format_time(total_time)}')
    print(f'   Dijalankan / diproses : {executed}')
    print(f'   Di-skip              : {skipped}')
    print(f'   Total hasil          : {len(df_results)}')
    print(f'   Live CSV             : {LIVE_RESULTS_PATH}')
    print(f'   Final CSV            : {FINAL_RESULTS_PATH}')
    print(f'{"=" * 80}')

    header = f"{'No':>3} | {'Eksperimen':<14} | {'F1 Test':>7} | {'F1 CV':>6} | {'Std':>6} | {'Gap':>6} | {'Fitur':<18} | Durasi"
    print(header)
    print('-' * len(header))
    for i, row in df_results.iterrows():
        feat_str = f"{int(row['n_feat_selected'])}/{int(row['n_feat_total'])} ({row['feat_ratio_pct']:.1f}%)"
        print(
            f"{i+1:>3} | {row['exp_id']:<14} | "
            f"{row['f1_test']:>7.4f} | {row['f1_cv_mean']:>6.4f} | {row['f1_cv_std']:>6.4f} | {row['f1_gap']:>6.4f} | "
            f"{feat_str:<18} | {format_time(row['duration_s'])}"
        )
    print('-' * len(header))

    display(df_results)


## 📊 Cell 8 — Ringkasan Per Dataset & Per Classifier


In [ ]:
if 'df_results' not in globals() or df_results.empty:
    print('⚠️ df_results belum tersedia atau masih kosong. Jalankan Cell 7 terlebih dahulu.')
else:
    print('📈 Rata-rata F1 Test per Dataset:')
    for ds, grp in df_results.groupby('dataset'):
        std_val = grp['f1_test'].std()
        std_val = 0.0 if pd.isna(std_val) else std_val
        print(f'   {ds:<10}: {grp["f1_test"].mean():.4f} ± {std_val:.4f}')

    print('\n🤖 Rata-rata F1 Test per Classifier:')
    for clf, grp in df_results.groupby('classifier'):
        std_val = grp['f1_test'].std()
        std_val = 0.0 if pd.isna(std_val) else std_val
        print(f'   {clf:<10}: {grp["f1_test"].mean():.4f} ± {std_val:.4f}')

    best = df_results.iloc[0]
    print(f'\n🏆 Terbaik : {best["exp_id"]}')
    print(f'   F1 Test  : {best["f1_test"]:.4f}')
    print(f'   F1 CV    : {best["f1_cv_mean"]:.4f} ± {best["f1_cv_std"]:.4f}')
    print(f'   Gap      : {best["f1_gap"]:.4f}')
    print(f'   Fitur    : {int(best["n_feat_selected"])}/{int(best["n_feat_total"])} ({best["feat_ratio_pct"]:.1f}%)')


## 🔧 Cell 9 — Regenerate Results dari Checkpoint


In [ ]:
# Jalankan cell ini hanya jika ga_results_live.csv hilang/rusak,
# tetapi checkpoint .pkl per eksperimen masih ada.

regen_rows = []

for ds_name, (X_tr, X_te, y_tr, y_te, subjects_tr) in datasets_to_run.items():
    dataset_dir = os.path.join(BASE_DIR, ds_name)
    if not os.path.exists(dataset_dir):
        continue

    for clf_name, clf in clf_to_run.items():
        exp_id = f'{ds_name}_{clf_name}'
        ckpt_file = os.path.join(dataset_dir, f'{exp_id}.pkl')
        if not os.path.exists(ckpt_file):
            print(f'⚠️  Tidak ada checkpoint: {exp_id}')
            continue

        try:
            state = load_state(ckpt_file)
        except Exception as e:
            print(f'❌ Gagal load checkpoint {exp_id}: {e}')
            continue

        if not state.get('done', False) or state.get('best_global_ind') is None:
            print(f'⏳ SKIP {exp_id} — checkpoint belum done')
            continue

        best_ind = state['best_global_ind']
        best_fit = float(state.get('best_global', 0.0))

        reset_cache()
        loso_folds = make_loso_folds(subjects_tr)
        fold_arrays = make_loso_fold_arrays(X_tr, y_tr, loso_folds)
        _ = fitness_function(best_ind, fold_arrays, clf, ALPHA, X_tr.shape[1])
        _, f1_cv_mean, f1_cv_std = get_cached_stats(best_ind)

        selected_idx = np.flatnonzero(best_ind)
        model_test = clone(clf)
        model_test.fit(X_tr[:, selected_idx], y_tr)
        pred_te = model_test.predict(X_te[:, selected_idx])
        f1_test = float(f1_score(y_te, pred_te, average='weighted'))

        regen_rows.append({
            'exp_id'          : exp_id,
            'dataset'         : ds_name,
            'classifier'      : clf_name,
            'cv_method'       : 'LeaveOneSubjectOut',
            'f1_test'         : round(f1_test, 4),
            'f1_cv_mean'      : round(f1_cv_mean, 4),
            'f1_cv_std'       : round(f1_cv_std, 4),
            'f1_gap'          : round(abs(f1_cv_mean - f1_test), 4),
            'best_fitness'    : round(best_fit, 4),
            'n_feat_selected' : len(selected_idx),
            'n_feat_total'    : X_tr.shape[1],
            'feat_ratio_pct'  : round(100 * len(selected_idx) / X_tr.shape[1], 2),
            'duration_s'      : round(float(state.get('elapsed_before', 0.0)), 2),
            'pop_size'        : POP_SIZE,
            'n_gen'           : N_GEN,
            'cr'              : CR,
            'mr'              : MR,
            'alpha'           : ALPHA,
        })
        print(f'✅ {exp_id} | F1 Test={f1_test:.4f} | F1 CV={f1_cv_mean:.4f} | Feat={len(selected_idx)}/{X_tr.shape[1]}')

if regen_rows:
    df_regen = pd.DataFrame(regen_rows).sort_values('f1_test', ascending=False).reset_index(drop=True)
    regen_path = os.path.join(BASE_DIR, 'ga_results_regenerated.csv')
    df_regen.to_csv(regen_path, index=False)
    print(f'\n💾 Regenerated results disimpan ke: {regen_path}')
    display(df_regen)
else:
    print('\n❌ Tidak ada checkpoint selesai yang bisa di-regenerate.')
